In [3]:


import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# URLs de tu repositorio de GitHub (formato Raw)
url_test = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/solicitudes_test.csv'
url_comportamiento = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/comportamiento_bancario.csv'
url_train = 'https://raw.githubusercontent.com/LucasAimetta/grupo3_aimetta_heinen_ia_2026/refs/heads/main/Competencia%20ML/solicitudes_train.csv'

# Cargar los datos directamente desde GitHub
df_solicitudes_test = pd.read_csv(url_test)
df_comportamiento = pd.read_csv(url_comportamiento)
df_solicitudes = pd.read_csv(url_train)


In [4]:
df_comportamiento['Mes_Referencia'] = pd.to_datetime(df_comportamiento['Mes_Referencia'])
df_solicitudes['Fecha_Solicitud'] = pd.to_datetime(df_solicitudes['Fecha_Solicitud'])
df_solicitudes_test['Fecha_Solicitud'] = pd.to_datetime(df_solicitudes_test['Fecha_Solicitud'])


def agregar_comportamiento(df_solicitudes_base, df_comportamiento):
    df_temp = pd.merge(
        left=df_solicitudes_base[['ID_Solicitud','ID_Usuario','Fecha_Solicitud']],
        right=df_comportamiento[['ID_Usuario','Mes_Referencia','Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']],
        on='ID_Usuario',
        how='left'
    )
    # Solo meses anteriores a CADA fecha de solicitud puntual
    df_temp = df_temp[df_temp['Mes_Referencia'] < df_temp['Fecha_Solicitud']]
    # HACEMOS PROMEDIO DE COMPORTAMIENTO
    return df_temp.groupby('ID_Solicitud')[['Saldo_Promedio_Mensual','Dias_Atraso_Otras_Deudas']].mean()

# Para train:
df_comportamiento_agrupado_train = agregar_comportamiento(df_solicitudes, df_comportamiento)
df_solicitudes_train_cruzado = pd.merge(df_solicitudes, df_comportamiento_agrupado_train, on='ID_Solicitud', how='left')

# Para test
df_comportamiento_agrupado_test = agregar_comportamiento(df_solicitudes_test, df_comportamiento)
df_solicitudes_test_cruzado = pd.merge(df_solicitudes_test, df_comportamiento_agrupado_test,on='ID_Solicitud',how='left')

df_solicitudes_train_cruzado.isnull().sum()


,0
ID_Solicitud,0
ID_Usuario,0
Monto_Solicitado,0
Tasa_Interes,0
Ingreso_Mensual,0
Motivo_Prestamo,0
Fecha_Solicitud,0
Estado_Final,0
Saldo_Promedio_Mensual,0
Dias_Atraso_Otras_Deudas,0


In [5]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

resultados_train = (df_solicitudes_train_cruzado["Estado_Final"])
arbol_train_base = pd.get_dummies(df_solicitudes_train_cruzado.drop(columns=["ID_Solicitud","Estado_Final", "ID_Usuario", "Fecha_Solicitud"]))
arbol_test_base = pd.get_dummies(df_solicitudes_test_cruzado.drop(columns=["ID_Solicitud", "ID_Usuario", "Fecha_Solicitud"]))

RS=70
arb_tr, arb_te, res_tr, res_te = train_test_split(arbol_train_base, resultados_train, test_size=0.3,stratify=resultados_train, random_state=RS)

modelo_boosting = GradientBoostingClassifier(n_estimators=25, max_depth=4,learning_rate=0.1, random_state=RS).fit(arbol_train_base, resultados_train)
print(f"AUC bagging (100 árboles): "
      f"{roc_auc_score(res_te, modelo_boosting.predict_proba(arb_te)[:, 1]):.3f}")



AUC bagging (100 árboles): 0.850


In [6]:

prediccion_binaria = modelo_boosting.predict(arbol_test_base)
prediccion_porcentaje = modelo_boosting.predict_proba(arbol_test_base)[:, 1]
df_solicitudes_test_cruzado["Prob_Pago"] = prediccion_porcentaje
df_solicitudes_test_cruzado['Esperanza']= (df_solicitudes_test_cruzado['Prob_Pago']*(df_solicitudes_test_cruzado['Monto_Solicitado'])*df_solicitudes_test_cruzado['Tasa_Interes'])-((1-df_solicitudes_test_cruzado['Prob_Pago'])*df_solicitudes_test_cruzado['Monto_Solicitado'])



In [7]:
#MECANISMO PARA NO PRESTAR DE MáS
monto_solicitado_total = 0
monto_a_ganar=0
#AñADIMOS UNA COLUMNA A LAS SOLICITUDES CON ESTADO = 0
df_solicitudes_test_cruzado['Estado'] = 0
#CREAMOS LISTA DE INDICES DE LAS FILAS DE LAS SOLICITUDES QUE APROBAREMOS
approved_indices = []

#ITERAMOS CADA FILA
for index, row in df_solicitudes_test_cruzado.sort_values(by='Prob_Pago', ascending=False).iterrows():
    # SI ES POSITIVA Y TENEMOS EL DINERO PARA PRESTAR, PRESTAMOS, GUARDAMOS INDICE Y ACTUALIZAMOS TOTAL PRESTADO
    if ((monto_solicitado_total + row['Monto_Solicitado']) < 2000000) and (row['Prob_Pago']>0.80):
        monto_solicitado_total += row['Monto_Solicitado']
        monto_a_ganar+= row['Tasa_Interes']*row['Monto_Solicitado']
        approved_indices.append(index)


# ACTUALIZAMOS EL ESTADO A 1, DE DICHAS SOLICITUDES QUE APROBAMOS
df_solicitudes_test_cruzado.loc[approved_indices, 'Estado'] = 1
monto_solicitado_total,monto_a_ganar


(1999700, 524950.85)

In [8]:
df_resultados = df_solicitudes_test_cruzado[['ID_Solicitud', 'Estado']].copy()
df_resultados.rename(columns={'Estado': 'Decision_Aprobacion'}, inplace=True)

# Exportar a CSV
df_resultados.to_csv('decisiones_aprobacion.csv', index=False)

print('Archivo decisiones_aprobacion.csv creado exitosamente.')
display(df_resultados.head())


Archivo decisiones_aprobacion.csv creado exitosamente.


,ID_Solicitud,Decision_Aprobacion
0,32417,0
1,32418,0
2,32419,0
3,32420,0
4,32421,1
